# Data Preprocessing - Simplified

This notebook merges preprocessed data and creates train/test splits.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.utils.logging_config import setup_logging
from src.utils.reproducibility import set_random_seeds

In [ ]:
# Setup
setup_logging()
set_random_seeds(42)

data_dir = Path('../data')
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

## 1. Load Preprocessed Data

In [ ]:
clinical_df = pd.read_csv(raw_dir / 'clinical_raw.csv')
genomic_df = pd.read_csv(raw_dir / 'genomic_raw.csv')
imaging_df = pd.read_csv(raw_dir / 'imaging_raw.csv')

print(f"Clinical: {clinical_df.shape}")
print(f"Genomic: {genomic_df.shape}")
print(f"Imaging: {imaging_df.shape}")

## 2. Add Modality Prefixes

In [ ]:
# Keep patientID without prefix for merging
patient_id = clinical_df['patientID'].copy()

# Add prefixes to distinguish modalities (except patientID)
clinical_cols = [c for c in clinical_df.columns if c != 'patientID']
genomic_cols = [c for c in genomic_df.columns if c != 'patientID']
imaging_cols = [c for c in imaging_df.columns if c != 'patientID']

clinical_prefixed = clinical_df[clinical_cols].add_prefix('clinical_')
genomic_prefixed = genomic_df[genomic_cols].add_prefix('genomic_')
imaging_prefixed = imaging_df[imaging_cols].add_prefix('imaging_')

# Add patientID back
clinical_prefixed['patientID'] = clinical_df['patientID']
genomic_prefixed['patientID'] = genomic_df['patientID']
imaging_prefixed['patientID'] = imaging_df['patientID']

print(f"Clinical features (prefixed): {len(clinical_cols)}")
print(f"Genomic features (prefixed): {len(genomic_cols)}")
print(f"Imaging features (prefixed): {len(imaging_cols)}")

## 3. Create Binary Target Labels from Diagnoses

In [ ]:
# Extract diagnosis descriptions from clinical data
# The diagnosisDescription column contains comma-separated diagnoses
diagnosis_col = 'clinical_diagnosisDescription'

# Create binary target columns based on diagnosis presence
def has_diagnosis(row, diagnosis_keyword):
    """Check if diagnosis keyword appears in patient's diagnosis list"""
    if pd.isna(row):
        return 0
    diagnoses_str = str(row).lower()
    return 1 if diagnosis_keyword.lower() in diagnoses_str else 0

# Create target columns
clinical_prefixed['hypertension'] = clinical_prefixed[diagnosis_col].apply(
    lambda x: has_diagnosis(x, 'hypertension')
)
clinical_prefixed['stroke'] = clinical_prefixed[diagnosis_col].apply(
    lambda x: has_diagnosis(x, 'stroke')
)
clinical_prefixed['coronary_heart_disease'] = clinical_prefixed[diagnosis_col].apply(
    lambda x: has_diagnosis(x, 'coronary') or has_diagnosis(x, 'heart disease')
)
clinical_prefixed['alzheimers_disease'] = clinical_prefixed[diagnosis_col].apply(
    lambda x: has_diagnosis(x, 'alzheimer')
)

print("\nTarget label distribution:")
print(f"Hypertension: {clinical_prefixed['hypertension'].sum()} / {len(clinical_prefixed)} ({clinical_prefixed['hypertension'].mean()*100:.1f}%)")
print(f"Stroke: {clinical_prefixed['stroke'].sum()} / {len(clinical_prefixed)} ({clinical_prefixed['stroke'].mean()*100:.1f}%)")
print(f"Coronary Heart Disease: {clinical_prefixed['coronary_heart_disease'].sum()} / {len(clinical_prefixed)} ({clinical_prefixed['coronary_heart_disease'].mean()*100:.1f}%)")
print(f"Alzheimer's Disease: {clinical_prefixed['alzheimers_disease'].sum()} / {len(clinical_prefixed)} ({clinical_prefixed['alzheimers_disease'].mean()*100:.1f}%)")

## 4. Merge Multi-Modal Data

In [ ]:
# Merge on patientID
multimodal_data = clinical_prefixed.merge(
    genomic_prefixed, on='patientID', how='inner'
).merge(
    imaging_prefixed, on='patientID', how='inner'
)

print(f"\nCombined data: {multimodal_data.shape}")
print(f"Patients: {multimodal_data['patientID'].nunique()}")
print(f"Total features: {len(multimodal_data.columns)}")

# Show sample of prefixed columns
print(f"\nSample clinical columns: {[c for c in multimodal_data.columns if c.startswith('clinical_')][:5]}")
print(f"Sample genomic columns: {[c for c in multimodal_data.columns if c.startswith('genomic_')][:5]}")
print(f"Sample imaging columns: {[c for c in multimodal_data.columns if c.startswith('imaging_')][:5]}")

# Verify target columns are present
target_columns = ['hypertension', 'stroke', 'coronary_heart_disease', 'alzheimers_disease']
print(f"\nTarget columns present: {all(t in multimodal_data.columns for t in target_columns)}")

## 5. Train/Test Split

In [ ]:
# Use hypertension for stratification (most common target)
train_data, test_data = train_test_split(
    multimodal_data,
    test_size=0.2,
    random_state=42,
    stratify=multimodal_data['hypertension']
)

print(f"\nTrain: {train_data.shape}")
print(f"Test: {test_data.shape}")

# Show target distribution in splits
print("\nTarget distribution in train set:")
for target in target_columns:
    print(f"  {target}: {train_data[target].sum()} / {len(train_data)} ({train_data[target].mean()*100:.1f}%)")

print("\nTarget distribution in test set:")
for target in target_columns:
    print(f"  {target}: {test_data[target].sum()} / {len(test_data)} ({test_data[target].mean()*100:.1f}%)")

## 6. Save Processed Data

In [ ]:
# Save full dataset
multimodal_data.to_csv(processed_dir / 'multimodal_features.csv', index=False)
print(f"✓ Saved multimodal_features.csv ({len(multimodal_data)} records)")

# Save train/test splits
train_data.to_csv(processed_dir / 'train_data.csv', index=False)
print(f"✓ Saved train_data.csv ({len(train_data)} records)")

test_data.to_csv(processed_dir / 'test_data.csv', index=False)
print(f"✓ Saved test_data.csv ({len(test_data)} records)")

# Save individual modalities (with prefixes)
clinical_prefixed.to_csv(processed_dir / 'clinical_features.csv', index=False)
genomic_prefixed.to_csv(processed_dir / 'genomic_features.csv', index=False)
imaging_prefixed.to_csv(processed_dir / 'imaging_features.csv', index=False)

print("\n✓ All processed data saved successfully!")